<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-13/notebooks/ClimatePipeline/01_ClimateDataDownloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateDataDownloader

Descargador genérico de observaciones climáticas IDEAM publicadas mediante Socrata. Guarda datos crudos en Parquet particionado por variable, fuente, departamento, año y mes.

Este notebook **solo descarga y normaliza tipos básicos**. No suma, promedia ni audita las observaciones, porque esas reglas dependen de cada variable climática.

## 1. Configuración

### Uso rápido

1. Escoja un `DATASET_ID` y un `VARIABLE_NOMBRE`.
2. Configure uno o ambos departamentos permitidos.
3. Configure listas de años y meses. Para todos los meses use `list(range(1, 13))`; para enero a abril use `[1, 2, 3, 4]`.
4. Mantenga `SOBRESCRIBIR_PARQUET = False` para reanudar descargas existentes.
5. Cambie `EJECUTAR_DESCARGA = True` y ejecute el notebook completo.

Fuentes candidatas:

| Variable | Dataset ID |
|---|---|
| Temperatura ambiente | `sbwg-7ju4` |
| Temperatura mínima | `afdg-3zpb` |
| Temperatura máxima | `ccvq-rp9s` |
| Humedad del aire | `uext-mhny` |
| Precipitación | `s54a-sgyg` |
| Velocidad del viento | `sgfv-3yp8` |
| Presión atmosférica | `62tk-nxj5` |

In [22]:
from pathlib import Path

import pandas as pd
import requests

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Fuente a descargar. Cambiar ambos valores juntos.
DATASET_ID = '62tk-nxj5'
VARIABLE_NOMBRE = 'presion_atmosferica'

# Es obligatorio indicar uno o ambos departamentos del alcance.
DESCARGA_DEPARTAMENTOS = [
    'CUNDINAMARCA',
]

# Ejemplos: [2025], [2024, 2025] o list(range(2019, 2026)).
DESCARGA_ANIOS = [2024]

# Ejemplos: [1, 2, 3, 4] o list(range(1, 13)).
DESCARGA_MESES = list(range(1, 13))

DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = None
MOSTRAR_CADA_N_LOTES = 25
SOBRESCRIBIR_PARQUET = False
REQUEST_TIMEOUT = 180
REQUEST_REINTENTOS = 3
APP_TOKEN = None  # Opcional. No subir tokens reales al repositorio.

# Banderita de seguridad para Run all.
EJECUTAR_DESCARGA = True

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)

print({
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': DESCARGA_DEPARTAMENTOS,
    'anios': DESCARGA_ANIOS,
    'meses': DESCARGA_MESES,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
    'mostrar_cada_n_lotes': MOSTRAR_CADA_N_LOTES,
    'sobrescribir': SOBRESCRIBIR_PARQUET,
    'ejecutar': EJECUTAR_DESCARGA,
    'processed_root': str(PROCESSED_ROOT),
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'dataset_id': '62tk-nxj5', 'variable': 'presion_atmosferica', 'departamentos': ['CUNDINAMARCA'], 'anios': [2024], 'meses': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'limit': 1000, 'max_lotes': None, 'mostrar_cada_n_lotes': 25, 'sobrescribir': False, 'ejecutar': True, 'processed_root': '/content/drive/MyDrive/eco2026_processed'}


## 2. Validación y plan de descarga

La configuración se valida antes de crear carpetas o consultar datos. Los departamentos se restringen al alcance aprobado: Boyacá y Cundinamarca.

In [23]:
import re
import unicodedata

DATASET_ID_PATTERN = re.compile(r'^[a-z0-9]{4}-[a-z0-9]{4}$', re.IGNORECASE)
DEPARTAMENTOS_PERMITIDOS = {'BOYACÁ', 'CUNDINAMARCA'}


def slugificar(valor):
    """Normaliza etiquetas para construir rutas deterministas."""
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('ascii').lower()
    texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
    if not texto:
        raise ValueError(f'No se pudo construir una etiqueta de ruta para {valor!r}.')
    return texto


def unicos_ordenados(valores):
    return sorted(set(valores))


def validar_configuracion():
    if not DATASET_ID_PATTERN.fullmatch(str(DATASET_ID)):
        raise ValueError('DATASET_ID debe tener el formato xxxx-xxxx.')
    if not str(VARIABLE_NOMBRE).strip():
        raise ValueError('VARIABLE_NOMBRE es obligatorio.')
    if not DESCARGA_DEPARTAMENTOS:
        raise ValueError('Debe configurar al menos un departamento.')

    departamentos = unicos_ordenados(str(d).strip().upper() for d in DESCARGA_DEPARTAMENTOS)
    no_permitidos = set(departamentos) - DEPARTAMENTOS_PERMITIDOS
    if no_permitidos:
        raise ValueError(
            f'Departamentos fuera del alcance: {sorted(no_permitidos)}. '
            f'Permitidos: {sorted(DEPARTAMENTOS_PERMITIDOS)}.'
        )

    if not DESCARGA_ANIOS:
        raise ValueError('DESCARGA_ANIOS no puede estar vacío.')
    anios = unicos_ordenados(int(a) for a in DESCARGA_ANIOS)
    if any(a < 1900 or a > 2100 for a in anios):
        raise ValueError(f'Años fuera de rango: {anios}.')

    if not DESCARGA_MESES:
        raise ValueError('DESCARGA_MESES no puede estar vacío.')
    meses = unicos_ordenados(int(m) for m in DESCARGA_MESES)
    if any(m < 1 or m > 12 for m in meses):
        raise ValueError(f'Los meses deben estar entre 1 y 12: {meses}.')

    if int(DESCARGA_LIMIT) <= 0:
        raise ValueError('DESCARGA_LIMIT debe ser positivo.')
    if DESCARGA_MAX_LOTES is not None and int(DESCARGA_MAX_LOTES) <= 0:
        raise ValueError('DESCARGA_MAX_LOTES debe ser None o un entero positivo.')
    if int(MOSTRAR_CADA_N_LOTES) <= 0:
        raise ValueError('MOSTRAR_CADA_N_LOTES debe ser un entero positivo.')

    return departamentos, anios, meses


def construir_plan(departamentos, anios, meses):
    return [
        {'departamento': departamento, 'anio': anio, 'mes': mes}
        for departamento in departamentos
        for anio in anios
        for mes in meses
    ]

## 3. Acceso a Socrata

Las consultas usan `LIMIT` y `OFFSET` dentro de `$query`. Cada solicitud tiene reintentos para errores transitorios y mantiene un orden estable por fecha, estación, sensor e ID interno.

In [24]:
import time


def headers_socrata():
    headers = {'Accept': 'application/json', 'User-Agent': 'RAIZ-ClimateDataDownloader/1.0'}
    if APP_TOKEN:
        headers['X-App-Token'] = APP_TOKEN
    return headers


def consultar_metadata(dataset_id):
    url = f'https://www.datos.gov.co/api/views/{dataset_id}'
    response = requests.get(
        url,
        headers=headers_socrata(),
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def validar_esquema_ideam(metadata):
    campos = {columna.get('fieldName') for columna in metadata.get('columns', [])}
    obligatorios = {'departamento', 'fechaobservacion'}
    faltantes = obligatorios - campos
    if faltantes:
        raise ValueError(
            f'El dataset no tiene el esquema climático esperado. Faltan: {sorted(faltantes)}.'
        )
    return campos


def consultar_lote(dataset_id, where, limit, offset, order, timeout=REQUEST_TIMEOUT):
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'
    query = (
        f'SELECT * WHERE {where} ORDER BY {order} '
        f'LIMIT {int(limit)} OFFSET {int(offset)}'
    )

    ultimo_error = None
    for intento in range(1, REQUEST_REINTENTOS + 1):
        try:
            response = requests.get(
                url,
                params={'$query': query},
                headers=headers_socrata(),
                timeout=timeout,
            )
            response.raise_for_status()
            return pd.DataFrame(response.json())
        except requests.RequestException as exc:
            ultimo_error = exc
            if intento == REQUEST_REINTENTOS:
                break
            espera = min(2 ** (intento - 1), 30)
            print(
                f'Consulta falló en intento {intento}/{REQUEST_REINTENTOS}: {exc}. '
                f'Reintentando en {espera} s...'
            )
            time.sleep(espera)

    raise ultimo_error


def construir_orden(campos):
    candidatos = ['fechaobservacion', 'codigoestacion', 'codigosensor']
    presentes = [campo for campo in candidatos if campo in campos]
    presentes.append(':id')
    return ', '.join(presentes)

## 4. Particiones y reanudación

Cada lote de hasta 1.000 filas se guarda como un archivo `part-xxxxx.parquet`. Todos los archivos de un mismo departamento, año y mes quedan en la misma carpeta.

### `SOBRESCRIBIR_PARQUET = False`

Es el modo recomendado. Si existen partes consecutivas, la descarga empieza en el siguiente índice y usa el `OFFSET` correspondiente. Si la partición ya estaba completa, la API devolverá cero filas y no se escribirá nada nuevo.

Antes de reanudar se revisa el número de filas de cada archivo. Todas las partes salvo la última deben tener exactamente `DESCARGA_LIMIT` filas. Si el último archivo tiene menos, la partición se considera completa. Esto también evita reanudar accidentalmente con un `LIMIT` diferente al usado en la primera corrida.

### `SOBRESCRIBIR_PARQUET = True`

La descarga vuelve a empezar en el lote cero y reemplaza archivos con el mismo nombre. No crea deliberadamente otra carpeta. Tampoco elimina partes antiguas sobrantes; por eso no se recomienda salvo que se entienda el estado de la partición.

`mkdir(..., exist_ok=True)` reutiliza la ruta montada. Si Google Drive muestra dos carpetas visualmente iguales, suele indicar rutas raíz distintas, accesos directos duplicados o diferencias invisibles en el nombre; no es efecto directo de esta bandera.

In [25]:
from datetime import datetime

PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')


def inicio_mes_siguiente(anio, mes):
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1


def ruta_particion(variable, dataset_id, departamento, anio, mes):
    return (
        PROCESSED_ROOT
        / 'clima_crudo'
        / f'variable={slugificar(variable)}'
        / f'fuente={dataset_id.lower()}'
        / f'departamento={slugificar(departamento)}'
        / f'anio={int(anio)}'
        / f'mes={int(mes):02d}'
    )


def partes_existentes(output_dir):
    encontrados = []
    for archivo in output_dir.glob('part-*.parquet'):
        coincidencia = PART_PATTERN.fullmatch(archivo.name)
        if coincidencia:
            encontrados.append((int(coincidencia.group(1)), archivo))
    encontrados.sort(key=lambda item: item[0])
    return encontrados


def validar_secuencia_partes(partes):
    indices = [indice for indice, _ in partes]
    if not indices:
        return
    esperados = list(range(indices[-1] + 1))
    if indices != esperados:
        faltantes = sorted(set(esperados) - set(indices))
        raise RuntimeError(
            f'La partición tiene huecos en sus archivos: {faltantes}. '
            'No se reanuda para evitar saltar registros.'
        )


def validar_filas_partes(partes, limit):
    if not partes:
        return [], False

    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError('Se necesita pyarrow para validar los Parquet existentes.') from exc

    filas = [pq.ParquetFile(archivo).metadata.num_rows for _, archivo in partes]
    inconsistentes = [
        indice
        for (indice, _), filas_archivo in zip(partes[:-1], filas[:-1])
        if filas_archivo != int(limit)
    ]
    if inconsistentes:
        raise RuntimeError(
            f'Las partes {inconsistentes} no tienen DESCARGA_LIMIT={limit} filas. '
            'No es seguro calcular el OFFSET para reanudar.'
        )
    if filas[-1] > int(limit):
        raise RuntimeError(
            f'La última parte tiene {filas[-1]} filas, más que el limit={limit}.'
        )

    particion_completa = filas[-1] < int(limit)
    return filas, particion_completa


def normalizar_lote(df, dataset_id):
    df = df.copy()
    if 'fechaobservacion' in df.columns:
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'], errors='coerce')
    for columna in ['valorobservado', 'latitud', 'longitud']:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')
    df['dataset_id'] = dataset_id
    return df


def descargar_particion(
    dataset_id,
    variable,
    departamento,
    anio,
    mes,
    campos,
    limit=1000,
    max_lotes=None,
    mostrar_cada_n_lotes=25,
    sobrescribir=False,
):
    output_dir = ruta_particion(variable, dataset_id, departamento, anio, mes)
    output_dir.mkdir(parents=True, exist_ok=True)

    existentes = partes_existentes(output_dir)
    validar_secuencia_partes(existentes)
    filas_existentes, ya_completa = validar_filas_partes(existentes, limit)
    inicio_idx = 0 if sobrescribir else len(existentes)
    lote_idx = inicio_idx
    offset = lote_idx * int(limit)
    lotes_consultados = 0
    filas_nuevas = 0
    bytes_nuevos = 0
    detalle = []
    inicio_tiempo = time.perf_counter()

    anio_fin, mes_fin = inicio_mes_siguiente(anio, mes)
    fecha_inicio = f'{int(anio):04d}-{int(mes):02d}-01T00:00:00'
    fecha_fin = f'{anio_fin:04d}-{mes_fin:02d}-01T00:00:00'
    departamento_sql = str(departamento).replace("'", "''")
    where = (
        f"departamento = '{departamento_sql}' "
        f"AND fechaobservacion >= '{fecha_inicio}' "
        f"AND fechaobservacion < '{fecha_fin}'"
    )
    order = construir_orden(campos)

    print(f'Carpeta: {output_dir}')
    print(f'Partes existentes: {len(existentes):,}')
    if filas_existentes:
        print(f'Filas por parte existente: {filas_existentes[:10]}')
    print(f'Inicio lote={lote_idx:,} | offset={offset:,} | sobrescribir={sobrescribir}')

    if ya_completa and not sobrescribir:
        print('La última parte tiene menos filas que el limit; la partición ya está completa.')
        resumen = {
            'dataset_id': dataset_id,
            'variable': slugificar(variable),
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'estado': 'ya_completa',
            'partes_existentes_inicio': len(existentes),
            'partes_escritas_corrida': 0,
            'filas_corrida': 0,
            'tamano_corrida_bytes': 0,
            'tamano_corrida_mb': 0.0,
            'duracion_segundos': round(time.perf_counter() - inicio_tiempo, 2),
            'carpeta': str(output_dir),
        }
        return resumen, pd.DataFrame()

    estado_particion = 'completa'
    ultimo_lote_escrito = None

    while True:
        if max_lotes is not None and lotes_consultados >= max_lotes:
            estado_particion = 'pausada_por_max_lotes'
            print(f'Corte por max_lotes={max_lotes}.')
            break

        reloj_inicio = datetime.now()
        lote_tiempo = time.perf_counter()
        df_lote = consultar_lote(
            dataset_id=dataset_id,
            where=where,
            limit=limit,
            offset=offset,
            order=order,
        )
        lotes_consultados += 1

        if df_lote.empty:
            print('La API no devolvió más filas para esta partición.')
            break

        df_lote = normalizar_lote(df_lote, dataset_id)
        output_path = output_dir / f'part-{lote_idx:05d}.parquet'
        ya_existia = output_path.exists()
        df_lote.to_parquet(output_path, index=False)

        filas_lote = len(df_lote)
        tamano_lote_bytes = output_path.stat().st_size
        filas_nuevas += filas_lote
        bytes_nuevos += tamano_lote_bytes
        ultimo_lote_escrito = lote_idx
        reloj_fin = datetime.now()
        duracion_lote = round(time.perf_counter() - lote_tiempo, 2)
        mostrar_progreso = (
            lotes_consultados == 1
            or lotes_consultados % int(mostrar_cada_n_lotes) == 0
            or filas_lote < limit
        )
        if mostrar_progreso:
            print(
                f'Lote {lote_idx:,} guardado | filas={filas_lote:,} | '
                f'tamaño={tamano_lote_bytes / 1024:.1f} KB | '
                f'duración={duracion_lote:.2f} s | fin={reloj_fin:%H:%M:%S}'
            )

        detalle.append({
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'lote': lote_idx,
            'offset': offset,
            'filas': filas_lote,
            'tamano_bytes': tamano_lote_bytes,
            'estado': 'sobrescrito' if ya_existia else 'escrito',
            'inicio': reloj_inicio.isoformat(timespec='seconds'),
            'fin': reloj_fin.isoformat(timespec='seconds'),
            'duracion_segundos': duracion_lote,
            'archivo': str(output_path),
        })

        if filas_lote < limit:
            print(f'Último lote detectado con {filas_lote:,} filas.')
            break

        lote_idx += 1
        offset += int(limit)

    if sobrescribir and existentes and ultimo_lote_escrito is not None:
        sobrantes = [indice for indice, _ in existentes if indice > ultimo_lote_escrito]
        if sobrantes:
            print(
                'ADVERTENCIA: quedaron partes antiguas con índices mayores al último '
                f'lote escrito: {sobrantes[:10]}.'
            )

    resumen = {
        'dataset_id': dataset_id,
        'variable': slugificar(variable),
        'departamento': departamento,
        'anio': int(anio),
        'mes': int(mes),
        'estado': estado_particion,
        'partes_existentes_inicio': len(existentes),
        'partes_escritas_corrida': len(detalle),
        'filas_corrida': filas_nuevas,
        'tamano_corrida_bytes': bytes_nuevos,
        'tamano_corrida_mb': round(bytes_nuevos / (1024 ** 2), 2),
        'duracion_segundos': round(time.perf_counter() - inicio_tiempo, 2),
        'carpeta': str(output_dir),
    }
    return resumen, pd.DataFrame(detalle)

## 5. Ejecución

El plan recorre todas las combinaciones configuradas. Un error en una partición queda registrado y no borra las partes descargadas anteriormente.

In [26]:
resumen_particiones = pd.DataFrame()
detalle_lotes = pd.DataFrame()

departamentos, anios, meses = validar_configuracion()
plan_descarga = construir_plan(departamentos, anios, meses)
vista_plan = pd.DataFrame(plan_descarga)
vista_plan['carpeta'] = vista_plan.apply(
    lambda fila: str(
        ruta_particion(
            VARIABLE_NOMBRE,
            DATASET_ID,
            fila['departamento'],
            fila['anio'],
            fila['mes'],
        )
    ),
    axis=1,
)
display(Markdown(f'## Plan configurado — {len(plan_descarga)} particiones'))
display(vista_plan)

if not EJECUTAR_DESCARGA:
    print('Descarga desactivada. Revise la configuración y cambie EJECUTAR_DESCARGA a True.')
else:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError('Para guardar Parquet instale pyarrow: !pip install pyarrow') from exc

    metadata = consultar_metadata(DATASET_ID)
    campos_dataset = validar_esquema_ideam(metadata)

    print(f"Dataset: {metadata.get('name')} ({DATASET_ID})")
    print(f'Variable de salida: {slugificar(VARIABLE_NOMBRE)}')
    print(f'Particiones a procesar: {len(plan_descarga):,}')
    print(f'Ruta raíz: {PROCESSED_ROOT / "clima_crudo"}')

    resumenes = []
    detalles = []

    for indice, item in enumerate(plan_descarga, start=1):
        titulo = (
            f"➡️ Partición {indice}/{len(plan_descarga)} — "
            f"{item['departamento']} | {item['anio']}-{item['mes']:02d}"
        )
        display(Markdown(f'---\n\n## {titulo}'))

        try:
            resumen, detalle = descargar_particion(
                dataset_id=DATASET_ID,
                variable=VARIABLE_NOMBRE,
                departamento=item['departamento'],
                anio=item['anio'],
                mes=item['mes'],
                campos=campos_dataset,
                limit=DESCARGA_LIMIT,
                max_lotes=DESCARGA_MAX_LOTES,
                mostrar_cada_n_lotes=MOSTRAR_CADA_N_LOTES,
                sobrescribir=SOBRESCRIBIR_PARQUET,
            )
            resumenes.append(resumen)
            if not detalle.empty:
                detalles.append(detalle)
        except Exception as exc:
            print(f'ERROR en partición: {type(exc).__name__}: {exc}')
            resumenes.append({
                'dataset_id': DATASET_ID,
                'variable': slugificar(VARIABLE_NOMBRE),
                **item,
                'estado': 'error',
                'error': f'{type(exc).__name__}: {exc}',
            })

    resumen_particiones = pd.DataFrame(resumenes)
    detalle_lotes = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()

    display(Markdown('---\n\n## Resumen final'))
    display(resumen_particiones)
    if not detalle_lotes.empty:
        display(Markdown('### Últimos 10 lotes de la corrida'))
        display(detalle_lotes.tail(10))

## Plan configurado — 12 particiones

,departamento,anio,mes,carpeta
0,CUNDINAMARCA,2024,1,/content/drive/MyDrive/eco2026_processed/clima...
1,CUNDINAMARCA,2024,2,/content/drive/MyDrive/eco2026_processed/clima...
2,CUNDINAMARCA,2024,3,/content/drive/MyDrive/eco2026_processed/clima...
3,CUNDINAMARCA,2024,4,/content/drive/MyDrive/eco2026_processed/clima...
4,CUNDINAMARCA,2024,5,/content/drive/MyDrive/eco2026_processed/clima...
5,CUNDINAMARCA,2024,6,/content/drive/MyDrive/eco2026_processed/clima...
6,CUNDINAMARCA,2024,7,/content/drive/MyDrive/eco2026_processed/clima...
7,CUNDINAMARCA,2024,8,/content/drive/MyDrive/eco2026_processed/clima...
8,CUNDINAMARCA,2024,9,/content/drive/MyDrive/eco2026_processed/clima...
9,CUNDINAMARCA,2024,10,/content/drive/MyDrive/eco2026_processed/clima...


Dataset: Presión Atmosférica (62tk-nxj5)
Variable de salida: presion_atmosferica
Particiones a procesar: 12
Ruta raíz: /content/drive/MyDrive/eco2026_processed/clima_crudo


---

## ➡️ Partición 1/12 — CUNDINAMARCA | 2024-01

Carpeta: /content/drive/MyDrive/eco2026_processed/clima_crudo/variable=presion_atmosferica/fuente=62tk-nxj5/departamento=cundinamarca/anio=2024/mes=01
Partes existentes: 0
Inicio lote=0 | offset=0 | sobrescribir=False
Lote 0 guardado | filas=1,000 | tamaño=19.0 KB | duración=0.43 s | fin=04:54:04
Lote 24 guardado | filas=1,000 | tamaño=19.7 KB | duración=0.64 s | fin=04:54:17
Lote 35 guardado | filas=811 | tamaño=17.7 KB | duración=0.62 s | fin=04:54:24
Último lote detectado con 811 filas.


...

Carpeta: /content/drive/MyDrive/eco2026_processed/clima_crudo/variable=presion_atmosferica/fuente=62tk-nxj5/departamento=cundinamarca/anio=2024/mes=12
Partes existentes: 0
Inicio lote=0 | offset=0 | sobrescribir=False
Lote 0 guardado | filas=1,000 | tamaño=12.7 KB | duración=0.53 s | fin=04:58:26
Lote 24 guardado | filas=1,000 | tamaño=15.3 KB | duración=0.78 s | fin=04:58:42
Lote 49 guardado | filas=1,000 | tamaño=20.2 KB | duración=0.96 s | fin=04:59:04
Lote 72 guardado | filas=884 | tamaño=14.8 KB | duración=1.19 s | fin=04:59:31
Último lote detectado con 884 filas.


---

## Resumen final

,dataset_id,variable,departamento,anio,mes,estado,partes_existentes_inicio,partes_escritas_corrida,filas_corrida,tamano_corrida_bytes,tamano_corrida_mb,duracion_segundos,carpeta
0,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,1,completa,0,36,35811,711200,0.68,20.66,/content/drive/MyDrive/eco2026_processed/clima...
1,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,2,completa,0,22,21924,425910,0.41,12.90,/content/drive/MyDrive/eco2026_processed/clima...
2,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,3,completa,0,20,19670,392427,0.37,11.37,/content/drive/MyDrive/eco2026_processed/clima...
3,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,4,completa,0,16,15830,319755,0.30,9.08,/content/drive/MyDrive/eco2026_processed/clima...
4,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,5,completa,0,20,19410,408916,0.39,13.80,/content/drive/MyDrive/eco2026_processed/clima...
5,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,6,completa,0,22,21983,459379,0.44,12.45,/content/drive/MyDrive/eco2026_processed/clima...
6,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,7,completa,0,24,23966,474613,0.45,17.00,/content/drive/MyDrive/eco2026_processed/clima...
7,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,8,completa,0,43,42574,840810,0.80,26.88,/content/drive/MyDrive/eco2026_processed/clima...
8,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,9,completa,0,33,32053,642365,0.61,21.13,/content/drive/MyDrive/eco2026_processed/clima...
9,62tk-nxj5,presion_atmosferica,CUNDINAMARCA,2024,10,completa,0,66,65417,1027458,0.98,52.67,/content/drive/MyDrive/eco2026_processed/clima...


### Últimos 10 lotes de la corrida

,departamento,anio,mes,lote,offset,filas,tamano_bytes,estado,inicio,fin,duracion_segundos,archivo
437,CUNDINAMARCA,2024,12,63,63000,1000,15362,escrito,2026-07-13T04:59:18,2026-07-13T04:59:19,1.05,/content/drive/MyDrive/eco2026_processed/clima...
438,CUNDINAMARCA,2024,12,64,64000,1000,15535,escrito,2026-07-13T04:59:19,2026-07-13T04:59:21,1.28,/content/drive/MyDrive/eco2026_processed/clima...
439,CUNDINAMARCA,2024,12,65,65000,1000,15655,escrito,2026-07-13T04:59:21,2026-07-13T04:59:22,1.19,/content/drive/MyDrive/eco2026_processed/clima...
440,CUNDINAMARCA,2024,12,66,66000,1000,15581,escrito,2026-07-13T04:59:22,2026-07-13T04:59:23,1.10,/content/drive/MyDrive/eco2026_processed/clima...
441,CUNDINAMARCA,2024,12,67,67000,1000,15497,escrito,2026-07-13T04:59:23,2026-07-13T04:59:25,1.68,/content/drive/MyDrive/eco2026_processed/clima...
442,CUNDINAMARCA,2024,12,68,68000,1000,15611,escrito,2026-07-13T04:59:25,2026-07-13T04:59:26,1.21,/content/drive/MyDrive/eco2026_processed/clima...
443,CUNDINAMARCA,2024,12,69,69000,1000,15600,escrito,2026-07-13T04:59:26,2026-07-13T04:59:27,1.13,/content/drive/MyDrive/eco2026_processed/clima...
444,CUNDINAMARCA,2024,12,70,70000,1000,15823,escrito,2026-07-13T04:59:27,2026-07-13T04:59:28,1.40,/content/drive/MyDrive/eco2026_processed/clima...
445,CUNDINAMARCA,2024,12,71,71000,1000,15661,escrito,2026-07-13T04:59:28,2026-07-13T04:59:30,1.41,/content/drive/MyDrive/eco2026_processed/clima...
446,CUNDINAMARCA,2024,12,72,72000,884,15158,escrito,2026-07-13T04:59:30,2026-07-13T04:59:31,1.19,/content/drive/MyDrive/eco2026_processed/clima...


In [27]:
if resumen_particiones.empty:
    print('No hay resultados de descarga para resumir.')
else:
    total_bytes = int(resumen_particiones['tamano_corrida_bytes'].fillna(0).sum())
    total_mb = total_bytes / (1024 ** 2)
    total_gb = total_bytes / (1024 ** 3)

    print('Tamaño total escrito en esta corrida:')
    print(f'- {total_bytes:,} bytes')
    print(f'- {total_mb:,.2f} MB')
    print(f'- {total_gb:,.4f} GB')

Tamaño total escrito en esta corrida:
- 7,958,212 bytes
- 7.59 MB
- 0.0074 GB


## 6. Qué queda pendiente

La salida conserva observaciones crudas normalizadas y trazabilidad de la fuente. Las siguientes tareas deben desarrollarse por variable y fuera de este notebook:

- Revisar duplicados por estación, sensor y fecha.
- Medir frecuencia y cobertura temporal por estación.
- Definir reglas de valores físicamente plausibles.
- Construir agregados diarios y por periodo agrícola.
- Decidir cómo combinar estaciones dentro de un municipio.

No use una suma genérica para temperatura, humedad, presión o viento.